# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pyrootutils
from pathlib import Path

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Parameters

In [3]:
N = 10
MAX_RECORDINGS = 200
CLIPS_PER_SPECIES = 2000
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

ACTIVE_COLLECTIONS = ["diff_species"]

# Select species

In [4]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
    select_diff_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more = True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species:   0%|          | 0/11167 [00:00<?, ?it/s]

Species pool: 1805 species with 100 XC recordings


In [5]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
Phylloscopus         32                 1                  1                 1
      Turdus         29                 1                  1                 1
   Setophaga         23                 1                  1                 1
    Emberiza         17                 1                  1                 1
       Vireo         15                 1                  1                 1
      Anthus         15                 1                  1                 1
Thamnophilus         14                 1                  1                 1
     Curruca         14                 1                  1                 1
Acrocephalus         12                 1                  1                 1
    Calidris         11                 1                  1                 1
       Strix         11                 1                  1                 1
      Trogon   

In [6]:
TARGET_GENUS  = "Phylloscopus"
TARGET_FAMILY = "Tyrannidae (Tyrant Flycatchers)"
TARGET_ORDER  = "Passeriformes"

In [7]:
collections: dict[str, list[str]] = {}

if "diff_species" in ACTIVE_COLLECTIONS:
    diff_species = select_same_genus(TARGET_GENUS, N, pool)
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    diff_genus = select_same_family(TARGET_FAMILY, N, pool)
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    diff_family = select_same_order(TARGET_ORDER, N, pool)
    collections["diff_family"] = [s.scientific_name for s in diff_family]
if "diff_order" in ACTIVE_COLLECTIONS:
    diff_order = select_diff_order(N, pool)
    collections["diff_order"] = [s.scientific_name for s in diff_order]

collections_to_use = {k: collections[k] for k in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_species:
  Phylloscopus collybita
  Phylloscopus trochilus
  Phylloscopus sibilatrix
  Phylloscopus inornatus
  Phylloscopus humei
  Phylloscopus ibericus
  Phylloscopus trochiloides
  Phylloscopus fuscatus
  Phylloscopus bonelli
  Phylloscopus xanthoschistos

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_species']


# Download from Xeno-canto

In [8]:
from building.download import download_species_list

all_species = list({name for names in collections.values() for name in names})
print(f"Downloading {len(all_species)} unique species ({MAX_RECORDINGS} recordings each)…")

downloaded = download_species_list(all_species, max_recordings=MAX_RECORDINGS)

print("\nDownload summary:")
for name, paths in downloaded.items():
    print(f"  {name}: {len(paths)} files")

  Phylloscopus collybita: 200 files


  Phylloscopus xanthoschistos: 200 files


  Phylloscopus trochiloides: 200 files


  Phylloscopus inornatus: 200 files


  Phylloscopus ibericus: 200 files


  Phylloscopus fuscatus: 200 files


  Phylloscopus humei: 200 files


  Phylloscopus trochilus: 200 files


  Phylloscopus sibilatrix: 200 files


  Phylloscopus bonelli: 200 files

Download summary:
  Phylloscopus collybita: 200 files
  Phylloscopus xanthoschistos: 200 files
  Phylloscopus trochiloides: 200 files
  Phylloscopus inornatus: 200 files
  Phylloscopus ibericus: 200 files
  Phylloscopus fuscatus: 200 files
  Phylloscopus humei: 200 files
  Phylloscopus trochilus: 200 files
  Phylloscopus sibilatrix: 200 files
  Phylloscopus bonelli: 200 files


# BirdNET validation + dataset assembly

In [9]:
from building.dataset import validate_and_build_all

dataset_paths = validate_and_build_all(
    collections_to_use,
    clips_per_species=CLIPS_PER_SPECIES,
    threshold=BIRDNET_THRESHOLD,
)

print("\nDatasets ready:")
for name, path in dataset_paths.items():
    print(f"  {name}: {path}")

Validating 10 species with BirdNET (threshold=0.92)...


Species:   0%|          | 0/10 [00:00<?, ?it/s]

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
read_audio_data


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


read_audio_data: complete, read  66 chunks.
analyze_recording XC723975.mp3


...

read_audio_data
read_audio_data: complete, read  93 chunks.
analyze_recording XC950181.mp3


read_audio_data


Species:  10%|█         | 1/10 [09:35<1:26:16, 575.20s/it]

read_audio_data: complete, read  12 chunks.
analyze_recording XC949645.mp3


KeyboardInterrupt: 

# Sanity check

In [ ]:
from pathlib import Path

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips  {species_counts}")